In [1]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
import sys
import re

def normalize_name(name):
    """Normalize name for matching"""
    if pd.isna(name):
        return ""
    # Remove extra whitespace, convert to lowercase
    name = str(name).strip().lower()
    # Remove common suffixes like Jr., Sr., III
    name = re.sub(r'\s+(jr\.?|sr\.?|iii|ii|iv)$', '', name, flags=re.IGNORECASE)
    return name

def parse_full_name(full_name):
    """Parse full name into first and last name"""
    if pd.isna(full_name) or full_name == "":
        return "", ""
    
    name_str = str(full_name).strip()
    parts = name_str.split()
    
    if len(parts) == 0:
        return "", ""
    elif len(parts) == 1:
        return parts[0], ""
    else:
        # First word is first name, rest is last name
        first_name = parts[0]
        last_name = ' '.join(parts[1:])
        return first_name, last_name

def find_salon_column(class_df):
    """Find the column index for Salón/room numbers"""
    # Check column names first
    for idx, col in enumerate(class_df.columns):
        col_name = str(col).lower()
        if 'salon' in col_name or 'salón' in col_name or 'room' in col_name or 'clase' in col_name:
            return idx
    
    # Look for a column with numbers like 701, 904, etc.
    for idx in range(len(class_df.columns)):
        sample = class_df.iloc[:, idx].dropna().head(20)
        # Check if values look like room numbers (3-4 digit numbers starting with 7, 8, 9, etc.)
        if sample.dtype in ['int64', 'float64', 'object']:
            numeric_count = sum(1 for x in sample if str(x).isdigit() and len(str(x)) >= 3)
            if numeric_count >= 10:
                return idx
    
    # Default to column 4 (index 3) if not found
    return 3

def find_class_for_name(first_name, last_name, class_df, salon_col_idx):
    """Find class for a given first and last name with fuzzy matching"""
    first_normalized = normalize_name(first_name)
    last_normalized = normalize_name(last_name)
    
    if not first_normalized and not last_normalized:
        return "No Name Provided"
    
    # Create normalized columns for matching
    class_df['first_norm'] = class_df.iloc[:, 0].apply(normalize_name)
    class_df['last_norm'] = class_df.iloc[:, 1].apply(normalize_name)
    
    # Try exact match first
    match = class_df[
        (class_df['first_norm'] == first_normalized) & 
        (class_df['last_norm'] == last_normalized)
    ]
    
    if not match.empty:
        salon = match.iloc[0, salon_col_idx]
        return str(int(salon)) if pd.notna(salon) else "Not Found"
    
    # Try matching by last name only if first name match fails
    if last_normalized:
        match = class_df[class_df['last_norm'] == last_normalized]
        if len(match) == 1:  # Only one match by last name
            salon = match.iloc[0, salon_col_idx]
            return str(int(salon)) if pd.notna(salon) else "Not Found"
    
    # Try matching by first name only
    if first_normalized:
        match = class_df[class_df['first_norm'] == first_normalized]
        if len(match) == 1:  # Only one match by first name
            salon = match.iloc[0, salon_col_idx]
            return str(int(salon)) if pd.notna(salon) else "Not Found"
    
    return "Not Found"

def detect_name_column(df):
    """Detect which column contains recipient names"""
    for col_idx, col in enumerate(df.columns):
        col_name = str(col).lower()
        if any(keyword in col_name for keyword in ['name', 'recipient', 'student']):
            return col_idx
    
    # If no obvious name column, look for the first text column with 2+ word entries
    for col_idx, col in enumerate(df.columns):
        if df[col].dtype == object:
            sample = df[col].dropna().head(10)
            multi_word = sum(1 for x in sample if len(str(x).split()) >= 2)
            if multi_word >= 5:
                return col_idx
    
    return 1  # Default to second column

def main(class_list_file, order_file, output_file):
    print("=" * 60)
    print("CLASS MATCHER - Matching orders to class list")
    print("=" * 60)
    
    # Read class list
    print(f"\n1. Reading class list from: {class_list_file}")
    class_df = pd.read_excel(class_list_file)
    print(f"   - Found {len(class_df)} students in class list")
    print(f"   - Columns: {class_df.columns.tolist()}")
    
    # Detect Salón column
    salon_col_idx = find_salon_column(class_df)
    salon_col_name = class_df.columns[salon_col_idx]
    print(f"   - Detected Salón column: '{salon_col_name}' (column {salon_col_idx})")
    print(f"   - Sample room numbers: {class_df.iloc[:5, salon_col_idx].tolist()}")
    
    # Read order file
    print(f"\n2. Reading orders from: {order_file}")
    order_df = pd.read_excel(order_file)
    print(f"   - Found {len(order_df)} orders")
    print(f"   - Columns: {order_df.columns.tolist()}")
    
    # Detect name column
    name_col_idx = detect_name_column(order_df)
    name_col = order_df.columns[name_col_idx]
    print(f"\n3. Detected name column: '{name_col}'")
    
    # Process each order
    print("\n4. Matching names to room numbers...")
    classes = []
    match_details = []
    
    for idx, row in order_df.iterrows():
        full_name = row[name_col]
        first_name, last_name = parse_full_name(full_name)
        student_class = find_class_for_name(first_name, last_name, class_df, salon_col_idx)
        classes.append(student_class)
        
        if student_class == "Not Found":
            match_details.append(f"   ❌ {full_name} -> Not Found")
        else:
            match_details.append(f"   ✓ {full_name} -> Room {student_class}")
    
    order_df['Salón'] = classes
    
    # Sort by recipient name
    print("\n5. Sorting by recipient name...")
    order_df = order_df.sort_values(by=name_col, key=lambda x: x.str.lower())
    
    # Statistics
    total_orders = len(order_df)
    found_count = (order_df['Salón'] != 'Not Found').sum()
    not_found_count = (order_df['Salón'] == 'Not Found').sum()
    
    print("\n" + "=" * 60)
    print("MATCHING RESULTS")
    print("=" * 60)
    print(f"Total orders: {total_orders}")
    print(f"Room numbers found: {found_count} ({found_count/total_orders*100:.1f}%)")
    print(f"Not found: {not_found_count} ({not_found_count/total_orders*100:.1f}%)")
    
    if not_found_count > 0:
        print("\nOrders without matches:")
        for detail in match_details:
            if "Not Found" in detail:
                print(detail)
    
    # Save to Excel with formatting
    print(f"\n6. Saving results to: {output_file}")
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        order_df.to_excel(writer, index=False, sheet_name='Orders with Salón')
        
        worksheet = writer.sheets['Orders with Salón']
        
        # Format headers (bold, centered, light gray background)
        header_fill = PatternFill(start_color='D3D3D3', end_color='D3D3D3', fill_type='solid')
        for cell in worksheet[1]:
            cell.font = Font(bold=True, size=11)
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.fill = header_fill
        
        # Highlight "Not Found" entries in red
        for row_idx in range(2, len(order_df) + 2):
            salon_cell = worksheet.cell(row=row_idx, column=order_df.columns.get_loc('Salón') + 1)
            if salon_cell.value == "Not Found":
                salon_cell.font = Font(color='FF0000', bold=True)
        
        # Auto-adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                if cell.value:
                    max_length = max(max_length, len(str(cell.value)))
            worksheet.column_dimensions[column_letter].width = min(max_length + 3, 50)
    
    print("\n✓ Processing complete!")
    print("=" * 60)

if __name__ == "__main__":
    if len(sys.argv) != 4:
        print("\n" + "=" * 60)
        print("CLASS MATCHER SCRIPT")
        print("=" * 60)
        print("\nUsage:")
        print("  python match_classes_enhanced.py <class_list.xlsx> <order_file.xlsx> <output_file.xlsx>")
        print("\nExample:")
        print("  python match_classes_enhanced.py class_list.xlsx orders.xlsx orders_with_classes.xlsx")
        print("\nThe script will:")
        print("  - Match recipient names to the class list")
        print("  - Add a 'Salón' column with room numbers (e.g., 701, 904)")
        print("  - Sort by recipient name")
        print("  - Highlight any names that couldn't be matched")
        print("=" * 60)
    else:
        try:
            main(sys.argv[1], sys.argv[2], sys.argv[3])
        except Exception as e:
            print(f"\n❌ Error: {e}")
            import traceback
            traceback.print_exc()

Done! File saved as: students_and_orders_with_salon.xlsx


/var/folders/wb/pmxl4r0s6svgkb08w2drl86c0000gn/T/ipykernel_18347/3270629371.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orders[name_col] = df_orders[name_col].astype(str).str.strip()
/var/folders/wb/pmxl4r0s6svgkb08w2drl86c0000gn/T/ipykernel_18347/3270629371.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orders[name_col] = df_orders[name_col].astype(str).str.strip()
/var/folders/wb/pmxl4r0s6svgkb08w2drl86c0000gn/T/ipykernel_18347/3270629371.py:55: SettingWithCopyWarning: 
A value is try

In [76]:
import pandas as pd

Classes = pd.read_excel("./students_and_orders.xlsx", sheet_name='Classes')

orders_8  = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 8' )[1:]
orders_9  = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 9' )[12:]
orders_10 = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 10')[12:]
orders_11 = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 11')[12:]
orders_12 = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 12')[10:]

for i in orders_9["Recipient Name"]:
    for j in range(0, len(Classes)):
        if i == Classes["FullName"].iloc[j]:
            class_row = Classes[Classes["FullName"] == i]
            print(class_row["Class"].iloc[0])

        orders_9.loc[orders_9["Recipient Name"] == i, "Grade"] = class_row["Class"].iloc[0]

# orders_11

orders_9

orders_9.to_excel("09.xlsx", index=False)

901.0
904.0
903.0
901.0
901.0
901.0
901.0
901.0
901.0
901.0
901.0
905.0
905.0
905.0
904.0
904.0
901.0
905.0
904.0
904.0
901.0
903.0
903.0
901.0
803.0
803.0
903.0
901.0
901.0
901.0
901.0
901.0
904.0
905.0
901.0
901.0
901.0
905.0
904.0
904.0
903.0
905.0
903.0
904.0
903.0
903.0
903.0
905.0


In [46]:
orders_8  = pd.read_excel("./students_and_orders.xlsx", sheet_name='Eton - Grade 8')[1:]

orders_8

,Gift,Recipient,Group,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
1,🍪 Galletas,Elisa barros,802.0,8,NO,Carmen Anaya,PAID,Anahuac-1770744002257-2327,NaN,NaN,NaN,NaN,NaN,Elisa barros,802,Elisa_Barros@eton.edu.mx,exacto: nombre+apellido
2,🍪 Brownies,Mariana Perez Garland,802.0,8,NO,Carmen Anaya,PAID,Anahuac-1770744002257-2327,NaN,NaN,NaN,NaN,NaN,Mariana Perez Garland,802,m_perez-garland@eton.edu.mx,exacto: nombre+apellido
3,🎶 Serenata,Elisa Gil,802.0,8,NO,Carmen Anaya,PAID,Anahuac-1770744002257-2327,NaN,NaN,NaN,NaN,NaN,Elisa Gil,802,elisa_gil@eton.edu.mx,exacto: nombre+apellido
4,🍪 Galletas,Julieta Martin Albarran,802.0,8,NO,Carmen Anaya,PAID,Anahuac-1770744002257-2327,NaN,NaN,NaN,NaN,NaN,Julieta Martin Albarran,802,julieta_martin@eton.edu.mx,exacto: nombre+apellidos
5,🍭 Paletas,Ana Paula Cartaya,802.0,8,NO,Carmen Anaya,PAID,Anahuac-1770744002257-2327,NaN,NaN,NaN,NaN,NaN,Ana Paula Cartaya,802,ana_cartaya@eton.edu.mx,exacto: nombre+apellido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,🌹 Roses,Gabriela Molina,805.0,805,NO,Mariana Perez,PAID,Eton-1770764585430-2839,NaN,NaN,NaN,NaN,NaN,Gabriela Molina,805,gabriela_molina@eton.edu.mx,exacto: nombre+apellido
370,🌹 Roses,Carmen Anaya,802.0,802,NO,Mariana Perez,PAID,Eton-1770764585430-2839,NaN,NaN,NaN,NaN,NaN,Carmen Anaya,802,carmen_anaya@eton.edu.mx,exacto: nombre+apellido
371,🌹 Roses,Ana Victoria Ros,802.0,802,NO,Mariana Perez,PAID,Eton-1770764585430-2839,NaN,NaN,NaN,NaN,NaN,Ana Victoria Ros,802,anavictoria_ros@eton.edu.mx,exacto: nombre+apellido
372,🌹 Roses,Valentina Torres,805.0,805,NO,Mariana Perez,PAID,Eton-1770764585430-2839,NaN,NaN,NaN,NaN,NaN,Valentina Torres,805,valentina_torres@eton.edu.mx,exacto: nombre+apellido


In [68]:
for i in orders_9["Recipient Name"]:
    for j in range(0, len(Classes)):
        if i == Classes["FullName"].iloc[j]:
            class_row = Classes[Classes["FullName"] == i]
            # print(class_row["Class"].iloc[0])

        orders_9.loc[orders_9["Recipient Name"] == i, "Grade"] = class_row["Class"].iloc[0]

gifts = {
    "Roses": 0,
    "Serenata": 0,
    "Galletas": 0,
    "Brownies": 0,
    "Paletas": 0,
    "Teddy": 0,
    "Brigadeiros": 0,
    "Globos": 0,
    "Chocolate Kiss": 0
}

c901 = []
c902 = []
c903 = []
c904 = []
c905 = []

for i in orders_9["Recipient Name"]:

    row = orders_9[orders_9["Recipient Name"] == i]

    if row["Grade"].iloc[0] == 901:
        c901.append(row)
    elif row["Grade"].iloc[0] == 902:
        c902.append(row)
    elif row["Grade"].iloc[0] == 903:
        c903.append(row)
    elif row["Grade"].iloc[0] == 904:
        c904.append(row)
    elif row["Grade"].iloc[0] == 905:
        c905.append(row)

for i in c901:
    for j in gifts.keys():
        # gifts[j] += i[j].iloc[0]

        print(j)
c902

Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brownies
Paletas
Teddy
Brigadeiros
Globos
Chocolate Kiss
Roses
Serenata
Galletas
Brow

[]

In [70]:
gifts = {
    "Roses": 0,
    "Serenata": 0,
    "Galletas": 0,
    "Brownies": 0,
    "Paletas": 0,
    "Teddy": 0,
    "Brigadeiros": 0,
    "Globos": 0,
    "Chocolate Kiss": 0
}

# for i in c901:
#     gift_value = i["Gift"].iloc[0]  # Get the gift string once
#     for j in gifts.keys():
#         if j in gift_value:
#             gifts[j] += 1  # Count once per order, not character count

# for i in c902:
#     gift_value = i["Gift"].iloc[0]
#     for j in gifts.keys():
#         if j in gift_value:
#             gifts[j] += 1

# for i in c903:
#     gift_value = i["Gift"].iloc[0]
#     for j in gifts.keys():
#         if j in gift_value:
#             gifts[j] += 1

# for i in c904:
#     gift_value = i["Gift"].iloc[0]
#     for j in gifts.keys():
#         if j in gift_value:
#             gifts[j] += 1

# for i in c905:
#     gift_value = i["Gift"].iloc[0]
#     for j in gifts.keys():
#         if j in gift_value:
#             gifts[j] += 1

all_classes = [c901, c902, c903, c904, c905]

for class_list in all_classes:
    for row_df in class_list:
        gift_value = row_df["Gift"].iloc[0]
        for gift_name in gifts.keys():
            if gift_name in gift_value:
                gifts[gift_name] += 1

gifts

{'Roses': 30,
 'Serenata': 19,
 'Galletas': 17,
 'Brownies': 56,
 'Paletas': 18,
 'Teddy': 1,
 'Brigadeiros': 11,
 'Globos': 2,
 'Chocolate Kiss': 13}

In [71]:
print(f"Total rows in orders_9: {len(orders_9)}")
print(f"Total in class lists: {len(c901) + len(c902) + len(c903) + len(c904) + len(c905)}")

Total rows in orders_9: 168
Total in class lists: 167


In [72]:
c901 = []
c902 = []
c903 = []
c904 = []
c905 = []

for idx, row in orders_9.iterrows():
    grade = row["Grade"]
    
    if grade == 901:
        c901.append(orders_9.loc[[idx]])  # Single row DataFrame
    elif grade == 902:
        c902.append(orders_9.loc[[idx]])
    elif grade == 903:
        c903.append(orders_9.loc[[idx]])
    elif grade == 904:
        c904.append(orders_9.loc[[idx]])
    elif grade == 905:
        c905.append(orders_9.loc[[idx]])

print(f"Total: {len(c901) + len(c902) + len(c903) + len(c904) + len(c905)}")

Total: 167


In [ ]:
import pandas as pd

# Load your file
file = "students.xlsx"

# Sheet with lookup names
df_main = pd.read_excel(file, sheet_name="orders")

# Sheet with full names + values
df_list = pd.read_excel(file, sheet_name="lista alumnos todo MS & HS")

# Columns:
# df_main["Name"] = short name
# df_list["Name"] = full name
# df_list["Class"] = value to return

def find_match(name):
    match = df_list[df_list["Name"].str.contains(name, case=False, na=False)]
    if len(match) > 0:
        return match.iloc[0]["Class"]   # first match
    return None

df_main["Class"] = df_main["Name"].apply(find_match)

# Save result
df_main.to_excel("output.xlsx", index=False)

In [78]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [79]:
import pandas as pd
from rapidfuzz import process

# ----------------------------
# 1. Load sheets (your setup)
# ----------------------------
Classes = pd.read_excel("./students_and_orders.xlsx", sheet_name="Classes")

orders_8  = pd.read_excel("./students_and_orders.xlsx", sheet_name="Eton - Grade 8")[1:]
orders_9  = pd.read_excel("./students_and_orders.xlsx", sheet_name="Eton - Grade 9")[12:]
orders_10 = pd.read_excel("./students_and_orders.xlsx", sheet_name="Eton - Grade 10")[12:]
orders_11 = pd.read_excel("./students_and_orders.xlsx", sheet_name="Eton - Grade 11")[12:]
orders_12 = pd.read_excel("./students_and_orders.xlsx", sheet_name="Eton - Grade 12")[10:]


# ----------------------------
# 2. Clean names (IMPORTANT)
# ----------------------------
Classes["FullName"] = (
    Classes["FullName"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Dictionary lookup (fast exact match)
lookup = dict(zip(Classes["FullName"], Classes["Class"]))
master_names = list(lookup.keys())


# ----------------------------
# 3. Function to fill grades
# ----------------------------
def fill_grades(df):

    # Clean recipient names
    df["Recipient Name"] = (
        df["Recipient Name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Treat 0 as missing
    df["Grade"] = df["Grade"].replace(0, pd.NA)

    # ---- Step 1: Exact match ----
    df["Grade"] = df["Grade"].fillna(
        df["Recipient Name"].map(lookup)
    )

    # ---- Step 2: Fuzzy fallback ----
    def fuzzy_lookup(name):
        best = process.extractOne(name, master_names)
        if best and best[1] >= 80:   # adjust threshold if needed
            return lookup[best[0]]
        return None

    mask = df["Grade"].isna()

    df.loc[mask, "Grade"] = (
        df.loc[mask, "Recipient Name"]
        .apply(fuzzy_lookup)
    )

    return df


# ----------------------------
# 4. Apply to all grades
# ----------------------------
orders_8  = fill_grades(orders_8)
orders_9  = fill_grades(orders_9)
orders_10 = fill_grades(orders_10)
orders_11 = fill_grades(orders_11)
orders_12 = fill_grades(orders_12)


# ----------------------------
# 5. Save final workbook
# ----------------------------
with pd.ExcelWriter("students_and_orders_FILLED.xlsx") as writer:
    Classes.to_excel(writer, sheet_name="Classes", index=False)

    orders_8.to_excel(writer, sheet_name="Eton - Grade 8", index=False)
    orders_9.to_excel(writer, sheet_name="Eton - Grade 9", index=False)
    orders_10.to_excel(writer, sheet_name="Eton - Grade 10", index=False)
    orders_11.to_excel(writer, sheet_name="Eton - Grade 11", index=False)
    orders_12.to_excel(writer, sheet_name="Eton - Grade 12", index=False)

print("Done → saved as students_and_orders_FILLED.xlsx")

KeyError: 'Grade'